# 05 — Build and understand a convolutional neural network

**Prerequisites:** notebooks 01–04: tensors, data splits, cross-entropy, and gradients.
**Goals:** trace image shapes, build a CNN, write its training loop, and inspect learned filters.
**Time:** about 20–35 minutes including exercises. The built-in 8×8 digits need no download.

Run cells in order. Read each prediction before running its code. The numbers you obtain depend
on hardware and random seeds; learning curves matter more than a particular score.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from course_utils import get_digits_loaders, evaluate

train_loader, val_loader, test_loader = get_digits_loaders(batch_size=64)
xb, yb = next(iter(train_loader))
print("Images N,C,H,W:", tuple(xb.shape), "| labels:", tuple(yb.shape))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, x, y in zip(axes, xb[:8], yb[:8]):
    ax.imshow(x[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(str(y.item())); ax.axis("off")
plt.show()


## 1. Follow the dimensions before training

A convolution slides the same small weight grid over the image: it learns a local pattern
at many positions. Channels represent different learned patterns, not separate images.
For dilation 1, one spatial output dimension is `floor((input + 2×padding − kernel)/stride) + 1`.
With a 3×3 kernel, stride 1, and padding 1, height and width stay unchanged.

Predict: `N×1×8×8 → N×16×8×8 → N×16×4×4 → N×32×4×4 → N×32×2×2 → N×128 → N×10`.
Pooling reduces spatial detail. ReLU introduces nonlinearity. The final layer returns **logits**;
`CrossEntropyLoss` expects logits and integer labels, so do not insert softmax before the loss.


In [ ]:
class DigitCNN(nn.Module):
    def __init__(self, width=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, width, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(width, width * 2, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((2, 2)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(width * 2 * 2 * 2, 10))

    def forward(self, x):
        return self.classifier(self.features(x))

model = DigitCNN().to(device)
z = xb[:2].to(device)
for layer in [*model.features, *model.classifier]:
    z = layer(z)
    print(f"{layer.__class__.__name__:20s} {tuple(z.shape)}")
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 2. Write the training loop yourself

Every batch performs: move data → predict → compute loss → clear old gradients → backpropagate
→ update parameters. `train()` enables training behavior; `eval()` selects inference behavior.
`inference_mode()` avoids constructing a gradient graph during validation.

We weight batch losses by batch size because the final batch may be smaller. Validation selects
the best epoch; the test set stays untouched until this model is selected.


In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_total, correct, count = 0.0, 0, 0
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = nn.functional.cross_entropy(logits, labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            loss_total += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            count += len(labels)
    return loss_total / count, correct / count

seed_everything(42)
model = DigitCNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)
epochs = 2 if SMOKE_TEST else 12
history, best_loss, best_state = [], float("inf"), None
for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader)
    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                        train_acc=train_acc, val_acc=val_acc))
    if val_loss < best_loss:
        best_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch:02d} | train {train_loss:.3f}/{train_acc:.1%} | val {val_loss:.3f}/{val_acc:.1%}")
model.load_state_dict(best_state)


## 3. Interpret training and validation together

Expect loss to decrease from roughly `log(10) ≈ 2.30`, although the initial value varies.
If training improves but validation worsens, suspect overfitting. If neither improves, check
image ranges, labels, shapes, learning rate, and whether `optimizer.step()` runs.
Smoke mode checks wiring only and is too short to judge accuracy.


In [ ]:
history_df = pd.DataFrame(history).set_index("epoch")
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
history_df[["train_loss", "val_loss"]].plot(ax=axes[0], title="Cross-entropy")
history_df[["train_acc", "val_acc"]].plot(ax=axes[1], title="Accuracy")
axes[1].set_ylim(0, 1); plt.tight_layout(); plt.show()
history_df.tail()


In [ ]:
# One final test evaluation after choosing the epoch from validation loss.
test_result = evaluate(model, test_loader, device=device)
print(f"Selected model test accuracy: {test_result['accuracy']:.1%}")
predictions = test_result["logits"].argmax(1)
mistakes = (predictions != test_result["targets"]).nonzero().flatten()
print("Test errors:", len(mistakes), "of", len(predictions))


## 4. Inspect filters and activations

The first-layer kernels are only 3×3, so do not expect each one to have an obvious human meaning.
Activations show where a learned filter responds in one image. These plots describe internal
calculations; they do not prove which visual evidence caused a decision.


In [ ]:
model.eval()
with torch.inference_mode():
    feature_maps = model.features[:2](xb[:1].to(device))[0].cpu()
kernels = model.features[0].weight.detach().cpu()
fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for j in range(8):
    axes[0, j].imshow(kernels[j, 0], cmap="coolwarm")
    axes[1, j].imshow(feature_maps[j], cmap="viridis")
    axes[0, j].set_title(f"Filter {j}")
    axes[0, j].axis("off"); axes[1, j].axis("off")
plt.tight_layout(); plt.show()


## 5. Save architecture information with weights

A `state_dict` stores parameters, not the Python class definition. Reload into the same
architecture. This checkpoint also records the expected image range, shape, and class order.


In [ ]:
checkpoint_path = ARTIFACTS / "05_digit_cnn.pt"
torch.save({"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "architecture": "DigitCNN(width=16)", "classes": [str(i) for i in range(10)],
            "input_shape": [1, 8, 8], "input_range": [0.0, 1.0]}, checkpoint_path)
print(checkpoint_path)


### Exercise: Change channel width

Predict how doubling width changes parameter count. Build both models; do not retrain yet.


In [ ]:
widths = [8, 16, 32]
parameter_counts = {w: sum(p.numel() for p in DigitCNN(w).parameters()) for w in widths}
print(parameter_counts)
# TODO: Explain why parameter count grows faster than width.


<details><summary>Sample answer — open after trying</summary>

The first convolution grows linearly with width, but the second connects `width` input channels to `2×width` outputs, so its weights grow quadratically. More parameters do not guarantee better validation accuracy.

</details>


### Exercise: Check softmax

Turn the logits for one batch into probabilities. Verify that each row sums to one, and compare predicted classes.


In [ ]:
with torch.inference_mode():
    example_logits = model(xb[:4].to(device)).cpu()
probabilities = example_logits.softmax(dim=1)
print("Probability sums:", probabilities.sum(1))
# TODO: Compare example_logits.argmax(1) and probabilities.argmax(1).


<details><summary>Sample answer — open after trying</summary>

```python
assert torch.equal(example_logits.argmax(1), probabilities.argmax(1))
```
Softmax preserves the largest element's index. Use probabilities for display and logits for cross-entropy.

</details>


### Exercise: Try a pooling alternative

Replace max pooling with average pooling in a fresh model. Optionally train it with the same seed and epoch budget; compare validation loss, without repeatedly consulting the test set.


In [ ]:
alternative = DigitCNN().to(device)
alternative.features[2] = nn.AvgPool2d(2)
print("Output shape:", alternative(xb[:2].to(device)).shape)
RUN_POOLING_EXPERIMENT = False
if RUN_POOLING_EXPERIMENT:
    from course_utils import fit
    seed_everything(42)
    alternative = DigitCNN().to(device)
    alternative.features[2] = nn.AvgPool2d(2)
    alternative_history = fit(alternative, train_loader, val_loader, epochs=epochs,
                              lr=0.003, weight_decay=1e-4, device=device)
    print(pd.DataFrame(alternative_history).tail())


<details><summary>Sample answer — open after trying</summary>

Max pooling retains the strongest response in each window; average pooling smooths responses. Reinitialize both comparisons with the same seed, reset data order, and compare several seeds before claiming one is better.

</details>


**Checkpoint:** explain `N,C,H,W`, predict each layer's shape, and describe one complete optimizer step.

**Optional next dataset:** after mastering the loop, try `torchvision.datasets.FashionMNIST`
(download required) with `ToTensor()`, a fixed stratified validation split from its training partition,
and a separate official test set. Adaptive pooling keeps this architecture compatible with 28×28 inputs;
update class names and saved preprocessing metadata. Do not mix its test images into training.

Next: [06 — Augmentation and regularization](06_augmentation_regularization.ipynb).
References: [Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html),
[cross-entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html),
[FashionMNIST](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html).
